In [3]:
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np
#Data import
df = pd.read_csv('../data/raw/alkanes_Stenutz.csv')

In [4]:
#add smiles
from src.utils.add_smiles import get_smiles
df['smiles'] = df.apply(lambda row: get_smiles(row['name']), axis=1)
df['smiles'] = df['smiles'].astype(str)
df

,name,number_ofC,molecular_weight,density,molar_volume,refractive_index,Molecular_refractive_power,dielectric_constant,melting_point,boiling_point,vapour_pressure,surface_tension,viscosity,logP,Tc,Pc,Vc,smiles
0,methane,1,16.04,0.424,37.8,1.0004,0.01,1.70,-183.00,-164,NaN,NaN,NaN,1.09,-82.0,45.6,99.0,C
1,ethane,2,30.07,0.546,55.1,1.2120,7.44,NaN,-182.00,-89,3.85,NaN,NaN,1.81,32.0,39.6,148.0,CC
2,propane,3,44.10,0.581,75.9,1.3400,15.90,1.60,-188.00,-45,NaN,NaN,NaN,2.36,97.0,42.0,203.0,CCC
3,butane,4,58.12,0.579,100.4,1.3560,21.95,1.77,-138.00,1,214.00,12.46,NaN,2.89,153.0,36.0,255.0,CCCC
4,isobutane,4,58.12,0.593,98.0,1.3520,21.20,NaN,-145.00,-12,304.00,NaN,NaN,2.76,135.0,36.0,263.0,CC(C)C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,"2,3,3-trimethylhexane",9,128.26,0.738,173.8,1.4140,43.44,NaN,-117.00,138,NaN,NaN,NaN,NaN,308.0,21.1,484.3,CCCC(C)(C)C(C)C
71,"2,3,4-trimethylhexane",9,128.26,0.739,173.5,1.4140,43.39,NaN,-116.79,139,NaN,NaN,NaN,NaN,311.0,22.0,483.5,CCC(C)C(C)C(C)C
72,"2,3,5-trimethylhexane",9,128.26,0.717,178.9,1.4050,43.85,NaN,-128.00,131,NaN,NaN,NaN,NaN,295.0,20.8,498.5,CC(C)CC(C)C(C)C
73,"2,4,4-trimethylhexane",9,128.26,0.724,177.2,1.4070,43.66,NaN,-113.00,131,NaN,NaN,NaN,NaN,295.0,20.6,493.8,CCC(C)(C)CC(C)C


In [5]:
#Calculate graph features: perron frobenius, information content, compression ratio, fiddler eigenvalue
from src.utils.graph_properties_calc import perron_frobenius,information_content,compression_ratio,fiedler_eigenvalue
# 1. Perron-Frobenius (Expects SMILES string -> Adjacency Matrix -> Eigenvalue)
df['perron_frobenius'] = df['smiles'].apply(perron_frobenius)

# 2. Information Content (Expects SMILES string -> Graph Symmetry -> Entropy)
df['information_content'] = df['smiles'].apply(information_content)

# 3. Compression Ratio (Expects SMILES string -> Compressed Bytes -> Ratio)
df['compression_ratio'] = df['smiles'].apply(compression_ratio)

# 4. Fiedler Eigenvalue (Expects SMILES string -> Laplacian Matrix -> Eigenvalue)
df['fiedler_eigenvalue'] = df['smiles'].apply(fiedler_eigenvalue)

df

,name,number_ofC,molecular_weight,density,molar_volume,refractive_index,Molecular_refractive_power,dielectric_constant,melting_point,boiling_point,...,viscosity,logP,Tc,Pc,Vc,smiles,perron_frobenius,information_content,compression_ratio,fiedler_eigenvalue
0,methane,1,16.04,0.424,37.8,1.0004,0.01,1.70,-183.00,-164,...,NaN,1.09,-82.0,45.6,99.0,C,0.000000,0.0000,0.0000,0.000000
1,ethane,2,30.07,0.546,55.1,1.2120,7.44,NaN,-182.00,-89,...,NaN,1.81,32.0,39.6,148.0,CC,1.000000,0.0000,0.5000,2.000000
2,propane,3,44.10,0.581,75.9,1.3400,15.90,1.60,-188.00,-45,...,NaN,2.36,97.0,42.0,203.0,CCC,1.414214,0.9183,0.6667,1.000000
3,butane,4,58.12,0.579,100.4,1.3560,21.95,1.77,-138.00,1,...,NaN,2.89,153.0,36.0,255.0,CCCC,1.618034,1.0000,0.5000,0.585786
4,isobutane,4,58.12,0.593,98.0,1.3520,21.20,NaN,-145.00,-12,...,NaN,2.76,135.0,36.0,263.0,CC(C)C,1.732051,0.8113,0.3333,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,"2,3,3-trimethylhexane",9,128.26,0.738,173.8,1.4140,43.44,NaN,-117.00,138,...,NaN,NaN,308.0,21.1,484.3,CCCC(C)(C)C(C)C,2.236068,2.7255,1.1667,0.223239
71,"2,3,4-trimethylhexane",9,128.26,0.739,173.5,1.4140,43.39,NaN,-116.79,139,...,NaN,NaN,311.0,22.0,483.5,CCC(C)C(C)C(C)C,2.164612,2.9477,1.5556,0.211786
72,"2,3,5-trimethylhexane",9,128.26,0.717,178.9,1.4050,43.85,NaN,-128.00,131,...,NaN,NaN,295.0,20.8,498.5,CC(C)CC(C)C(C)C,2.116883,2.7255,1.1667,0.183044
73,"2,4,4-trimethylhexane",9,128.26,0.724,177.2,1.4070,43.66,NaN,-113.00,131,...,NaN,NaN,295.0,20.6,493.8,CCC(C)(C)CC(C)C,2.193993,2.7255,1.1667,0.204260


In [6]:
THRESHOLD = 0.90
availability_rates = df.count() / len(df)
df = df.loc[:, availability_rates > THRESHOLD]
df

,name,number_ofC,molecular_weight,density,molar_volume,refractive_index,Molecular_refractive_power,melting_point,boiling_point,Tc,Pc,Vc,smiles,perron_frobenius,information_content,compression_ratio,fiedler_eigenvalue
0,methane,1,16.04,0.424,37.8,1.0004,0.01,-183.00,-164,-82.0,45.6,99.0,C,0.000000,0.0000,0.0000,0.000000
1,ethane,2,30.07,0.546,55.1,1.2120,7.44,-182.00,-89,32.0,39.6,148.0,CC,1.000000,0.0000,0.5000,2.000000
2,propane,3,44.10,0.581,75.9,1.3400,15.90,-188.00,-45,97.0,42.0,203.0,CCC,1.414214,0.9183,0.6667,1.000000
3,butane,4,58.12,0.579,100.4,1.3560,21.95,-138.00,1,153.0,36.0,255.0,CCCC,1.618034,1.0000,0.5000,0.585786
4,isobutane,4,58.12,0.593,98.0,1.3520,21.20,-145.00,-12,135.0,36.0,263.0,CC(C)C,1.732051,0.8113,0.3333,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,"2,3,3-trimethylhexane",9,128.26,0.738,173.8,1.4140,43.44,-117.00,138,308.0,21.1,484.3,CCCC(C)(C)C(C)C,2.236068,2.7255,1.1667,0.223239
71,"2,3,4-trimethylhexane",9,128.26,0.739,173.5,1.4140,43.39,-116.79,139,311.0,22.0,483.5,CCC(C)C(C)C(C)C,2.164612,2.9477,1.5556,0.211786
72,"2,3,5-trimethylhexane",9,128.26,0.717,178.9,1.4050,43.85,-128.00,131,295.0,20.8,498.5,CC(C)CC(C)C(C)C,2.116883,2.7255,1.1667,0.183044
73,"2,4,4-trimethylhexane",9,128.26,0.724,177.2,1.4070,43.66,-113.00,131,295.0,20.6,493.8,CCC(C)(C)CC(C)C,2.193993,2.7255,1.1667,0.204260


In [7]:
df_numeric = df.drop(columns=['name', 'smiles'])
df_numeric

,number_ofC,molecular_weight,density,molar_volume,refractive_index,Molecular_refractive_power,melting_point,boiling_point,Tc,Pc,Vc,perron_frobenius,information_content,compression_ratio,fiedler_eigenvalue
0,1,16.04,0.424,37.8,1.0004,0.01,-183.00,-164,-82.0,45.6,99.0,0.000000,0.0000,0.0000,0.000000
1,2,30.07,0.546,55.1,1.2120,7.44,-182.00,-89,32.0,39.6,148.0,1.000000,0.0000,0.5000,2.000000
2,3,44.10,0.581,75.9,1.3400,15.90,-188.00,-45,97.0,42.0,203.0,1.414214,0.9183,0.6667,1.000000
3,4,58.12,0.579,100.4,1.3560,21.95,-138.00,1,153.0,36.0,255.0,1.618034,1.0000,0.5000,0.585786
4,4,58.12,0.593,98.0,1.3520,21.20,-145.00,-12,135.0,36.0,263.0,1.732051,0.8113,0.3333,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,9,128.26,0.738,173.8,1.4140,43.44,-117.00,138,308.0,21.1,484.3,2.236068,2.7255,1.1667,0.223239
71,9,128.26,0.739,173.5,1.4140,43.39,-116.79,139,311.0,22.0,483.5,2.164612,2.9477,1.5556,0.211786
72,9,128.26,0.717,178.9,1.4050,43.85,-128.00,131,295.0,20.8,498.5,2.116883,2.7255,1.1667,0.183044
73,9,128.26,0.724,177.2,1.4070,43.66,-113.00,131,295.0,20.6,493.8,2.193993,2.7255,1.1667,0.204260


In [8]:
# --- 1. Define Features and Target ---
# We use only the structural/topological columns that can be derived from SMILES
calculable_cols = [
    'perron_frobenius', 'information_content', 'fiedler_eigenvalue',
    'compression_ratio', 'molecular_weight', 'number_ofC'
]

# Ensure we only use columns that exist in your dataframe
available_cols = [c for c in calculable_cols if c in df_numeric.columns]

target_var = 'Vc'

X_qspr = df_numeric[available_cols]
y_qspr = df_numeric[target_var] # Target variable (Critical Temperature)

# --- 2. Rigorous Data Partitioning (80/20 Split) ---
# CRITICAL: This prevents data leakage and proves your model generalizes
X_train, X_test, y_train, y_test = train_test_split(
    X_qspr, y_qspr, test_size=0.2, random_state=42
)

# --- 3. Strict Feature Standardization ---
# Fit the scaler ONLY on the training data, then transform both
scaler_qspr = StandardScaler()
X_train_scaled = scaler_qspr.fit_transform(X_train)
X_test_scaled = scaler_qspr.transform(X_test)

# --- 4. Fit the QSPR Model (Linear Regression) ---
# We fit directly on the scaled features, NOT kPCA, to preserve interpretability
reg_qspr = LinearRegression()
reg_qspr.fit(X_train_scaled, y_train)

# --- 5. Empirical Validation ---
y_pred = reg_qspr.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"----Direct Regression----")
print(f"Testing R-squared: {r2:.4f}")
print(f"Testing RMSE: {rmse:.2f} C")

# --- 6. Feature Importance Extraction ---
# This gives the exact weights linking symmetry to physical properties
importances = pd.DataFrame({
    'Feature': available_cols,
    'Coefficient': reg_qspr.coef_
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("\n--- Feature Importances ---")
print(importances)

def predict_alkane(smiles_string):
    mol = Chem.MolFromSmiles(smiles_string)
    if not mol: return "Invalid SMILES"

    data = {
        'perron_frobenius': perron_frobenius(smiles_string),
        'information_content': information_content(smiles_string),
        'fiedler_eigenvalue': fiedler_eigenvalue(smiles_string),
        'compression_ratio': compression_ratio(smiles_string),
        'molecular_weight': Descriptors.MolWt(mol),
        'number_ofC': sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')
    }

    # Create DataFrame and enforce strict column order to match training
    df_single = pd.DataFrame([data])
    df_single = df_single[available_cols]

    # Scale using the training scaler
    X_single_scaled = scaler_qspr.transform(df_single)

    # Predict directly (No kPCA)
    prediction = reg_qspr.predict(X_single_scaled)[0]

    return prediction

# --- FINAL TEST ---
test_smiles = "CCCCCCCCC(C)C" # Decane -> does not exist in starting dataset
predicted_value = predict_alkane(test_smiles)

print(f"\nMolecule: {test_smiles}")
print(f"Predicted {target_var}: {predicted_value:.2f} ")

----Direct Regression----
Testing R-squared: 0.7990
Testing RMSE: 48.36 C

--- Feature Importances ---
               Feature   Coefficient
5           number_ofC  79326.140701
4     molecular_weight -79252.697196
2   fiedler_eigenvalue    -13.915473
0     perron_frobenius    -13.766543
3    compression_ratio     -3.470207
1  information_content     -0.501305

Molecule: CCCCCCCCC(C)C
Predicted Vc: 603.36 


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from rdkit.Chem import Descriptors
from rdkit import Chem
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error, r2_score

correlations = []
for i in range(X_train_kpca_3d.shape[1]):
    corr = np.corrcoef(X_train_kpca_3d[:, i], y_train)[0, 1]
    correlations.append(abs(corr))

# 2. Automatically grab the indices of the top 3
best_3_indices = np.argsort(correlations)[-3:][::-1]
col_names = [f'kPCA_{i+1}' for i in best_3_indices]

calculable_cols = [
    'perron_frobenius', 'information_content', 'fiedler_eigenvalue',
    'compression_ratio', 'molecular_weight', 'number_ofC'
]
available_cols = [c for c in calculable_cols if c in df_numeric.columns]

X = df_numeric[available_cols]
y = df_numeric['Tc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_kpca = kpca_3d.fit_transform(X_train_scaled)
X_test_kpca = kpca_3d.transform(X_test_scaled)

reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train_kpca, y_train)

y_pred = reg_model.predict(X_test_kpca)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n--- Model Performance (kPCA + Random Forest) ---")
print(f"RMSE: {rmse:.2f} °C")

# --- PREDICTION FUNCTION FOR NEW ALKANES ---
def predict_tc_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    feat_values = {
        'perron_frobenius': perron_frobenius(smiles),
        'information_content': information_content(smiles),
        'fiedler_eigenvalue': fiedler_eigenvalue(smiles),
        'compression_ratio': compression_ratio(smiles),
        'molecular_weight': Descriptors.MolWt(mol),
        'number_ofC': sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')
    }

    df_new = pd.DataFrame([feat_values])[available_cols]
    new_scaled = scaler.transform(df_new)
    new_kpca = kpca_3d.transform(new_scaled)

    # Prediction
    prediction = reg_model.predict(new_kpca)[0]
    return prediction

# Test
example_smiles = "CCCCCCCCCC" # Decane
predicted_val = predict_tc_from_smiles(example_smiles)
print(f"\nFinal Prediction for {example_smiles}: {predicted_val:.2f} °C")